In [1]:
# ============================================================
# OPENLENS:
# LOKALE EMBEDDINGS AUS DEN TEXT-CHUNKS ERZEUGEN
# ============================================================
#
# Dieser Code:
# - erkennt das OpenLens-Projekt automatisch
# - lädt die erzeugten Text-Chunks
# - installiert bei Bedarf sentence-transformers
# - lädt beim ersten Lauf ein mehrsprachiges Embedding-Modell
# - speichert das Modell anschließend lokal im Projekt
# - erzeugt normalisierte Embeddings für alle Chunks
# - speichert die Vektoren als NumPy-Datei
# - speichert die zugehörigen Chunk-Metadaten
# - erstellt eine Manifest-Datei
# - führt eine erste semantische Testsuche durch
#
# Beim ersten Start ist Internet für den Modelldownload notwendig.
# Danach wird das lokal gespeicherte Modell verwendet.
# ============================================================


# ============================================================
# 1. STANDARD-BIBLIOTHEKEN IMPORTIEREN
# ============================================================

import json
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# ============================================================
# 2. SENTENCE-TRANSFORMERS PRÜFEN UND BEI BEDARF INSTALLIEREN
# ============================================================

try:
    from sentence_transformers import SentenceTransformer

except ImportError:

    print("sentence-transformers ist noch nicht installiert.")
    print("Die Installation wird jetzt gestartet ...")

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--upgrade",
            "sentence-transformers",
        ]
    )

    from sentence_transformers import SentenceTransformer

    print("sentence-transformers wurde erfolgreich installiert.")


# ============================================================
# 3. EINSTELLUNGEN
# ============================================================

# Mehrsprachiges Modell für deutsche und internationale Texte
MODEL_NAME = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

# Anzahl der Chunks, die gleichzeitig verarbeitet werden.
# Bei wenig Arbeitsspeicher auf 16 reduzieren.
BATCH_SIZE = 32

# Fortschrittsanzeige während der Berechnung
SHOW_PROGRESS_BAR = True

# Bereits vorhandene Embeddings neu erzeugen?
#
# False:
# Vorhandene Dateien werden verwendet.
#
# True:
# Alle Embeddings werden neu berechnet.
FORCE_REBUILD = False

# Suchanfrage für den Funktionstest am Ende
TEST_QUERY = (
    "Dokumente über Informationsfreiheit, "
    "Behörden und staatliche Entscheidungen"
)

# Anzahl der Suchergebnisse im Funktionstest
TOP_K = 10


# ============================================================
# 4. PROJEKTORDNER AUTOMATISCH ERKENNEN
# ============================================================

ARBEITSORDNER = Path.cwd().resolve()

BEKANNTE_UNTERORDNER = {
    "datenbank",
    "dokumente",
    "texte",
    "bereinigte_texte",
    "chunks",
    "embeddings",
    "modelle",
}


if ARBEITSORDNER.name.lower() in BEKANNTE_UNTERORDNER:

    PROJEKTORDNER = ARBEITSORDNER.parent

elif (ARBEITSORDNER / "Datenbank").is_dir():

    PROJEKTORDNER = ARBEITSORDNER

else:

    raise FileNotFoundError(
        "\nDer OpenLens-Projektordner konnte nicht erkannt werden.\n\n"
        f"Aktueller Arbeitsordner:\n{ARBEITSORDNER}\n\n"
        "Erwartet wurde entweder der OpenLens-Hauptordner "
        "oder einer seiner Unterordner."
    )


DATENBANKORDNER = PROJEKTORDNER / "Datenbank"
CHUNK_ORDNER = PROJEKTORDNER / "Chunks"
EMBEDDING_ORDNER = PROJEKTORDNER / "Embeddings"
MODELL_ORDNER = PROJEKTORDNER / "Modelle"

EMBEDDING_ORDNER.mkdir(
    parents=True,
    exist_ok=True,
)

MODELL_ORDNER.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 5. DATEIPFADE FESTLEGEN
# ============================================================

CHUNKS_JSONL = (
    CHUNK_ORDNER
    / "fragdenstaat_chunks.jsonl"
)

CHUNKS_CSV = (
    CHUNK_ORDNER
    / "fragdenstaat_chunks.csv"
)

LOKALES_MODELL = (
    MODELL_ORDNER
    / "paraphrase-multilingual-MiniLM-L12-v2"
)

EMBEDDINGS_NPY = (
    EMBEDDING_ORDNER
    / "fragdenstaat_embeddings.npy"
)

EMBEDDING_METADATA_JSONL = (
    EMBEDDING_ORDNER
    / "fragdenstaat_embedding_metadata.jsonl"
)

EMBEDDING_METADATA_CSV = (
    EMBEDDING_ORDNER
    / "fragdenstaat_embedding_metadata.csv"
)

EMBEDDING_MANIFEST = (
    EMBEDDING_ORDNER
    / "fragdenstaat_embedding_manifest.json"
)

SUCHERGEBNIS_TEST_CSV = (
    EMBEDDING_ORDNER
    / "semantic_search_test.csv"
)


print("=" * 80)
print("OPENLENS: LOKALE EMBEDDINGS ERZEUGEN")
print("=" * 80)

print("\nArbeitsordner:")
print(ARBEITSORDNER)

print("\nProjektordner:")
print(PROJEKTORDNER)

print("\nChunk-Ordner:")
print(CHUNK_ORDNER)

print("\nEmbedding-Ordner:")
print(EMBEDDING_ORDNER)

print("\nLokaler Modellordner:")
print(LOKALES_MODELL)


# ============================================================
# 6. CHUNK-DATEN LADEN
# ============================================================

if CHUNKS_JSONL.exists():

    print("\nChunk-Daten werden aus JSONL geladen ...")

    chunks_df = pd.read_json(
        CHUNKS_JSONL,
        lines=True,
    )

    verwendete_chunk_datei = CHUNKS_JSONL

elif CHUNKS_CSV.exists():

    print("\nJSONL nicht gefunden. Chunk-Daten werden aus CSV geladen ...")

    chunks_df = pd.read_csv(
        CHUNKS_CSV,
        encoding="utf-8-sig",
        low_memory=False,
    )

    verwendete_chunk_datei = CHUNKS_CSV

else:

    raise FileNotFoundError(
        "\nEs wurden keine Chunk-Daten gefunden.\n\n"
        "Erwartete Dateien:\n"
        f"- {CHUNKS_JSONL}\n"
        f"- {CHUNKS_CSV}\n\n"
        "Führe zuerst das Notebook "
        "'05_Clean_and_Chunk_Text.ipynb' aus."
    )


print("\nGeladene Chunk-Datei:")
print(verwendete_chunk_datei)

print("\nGeladene Chunks:")
print(f"{len(chunks_df):,}")


# ============================================================
# 7. NOTWENDIGE SPALTEN PRÜFEN
# ============================================================

PFLICHTSPALTEN = [
    "chunk_id",
    "document_id",
    "text",
]

fehlende_spalten = [
    spalte
    for spalte in PFLICHTSPALTEN
    if spalte not in chunks_df.columns
]

if fehlende_spalten:

    raise KeyError(
        "\nFolgende notwendige Chunk-Spalten fehlen:\n"
        + "\n".join(
            f"- {spalte}"
            for spalte in fehlende_spalten
        )
    )


# ============================================================
# 8. CHUNK-DATEN BEREINIGEN
# ============================================================

urspruengliche_chunk_anzahl = len(
    chunks_df
)

chunks_df["chunk_id"] = (
    chunks_df["chunk_id"]
    .fillna("")
    .astype(str)
    .str.strip()
)

chunks_df["text"] = (
    chunks_df["text"]
    .fillna("")
    .astype(str)
    .str.strip()
)

chunks_df["document_id"] = pd.to_numeric(
    chunks_df["document_id"],
    errors="coerce",
)


# Leere Texte entfernen
chunks_df = chunks_df[
    chunks_df["text"].ne("")
].copy()


# Ungültige Dokument-IDs entfernen
chunks_df = chunks_df[
    chunks_df["document_id"].notna()
].copy()

chunks_df["document_id"] = (
    chunks_df["document_id"]
    .astype("int64")
)


# Leere Chunk-IDs entfernen
chunks_df = chunks_df[
    chunks_df["chunk_id"].ne("")
].copy()


# Doppelte Chunk-IDs entfernen
anzahl_vor_duplikaten = len(
    chunks_df
)

chunks_df = (
    chunks_df
    .drop_duplicates(
        subset=["chunk_id"],
        keep="last",
    )
    .reset_index(drop=True)
)

entfernte_duplikate = (
    anzahl_vor_duplikaten
    - len(chunks_df)
)

insgesamt_entfernt = (
    urspruengliche_chunk_anzahl
    - len(chunks_df)
)


if chunks_df.empty:

    raise ValueError(
        "\nNach der Bereinigung sind keine nutzbaren Chunks übrig."
    )


print("\nChunk-Bereinigung:")
print("-" * 60)
print(
    "Ursprüngliche Chunks:",
    f"{urspruengliche_chunk_anzahl:,}",
)
print(
    "Entfernte doppelte Chunk-IDs:",
    f"{entfernte_duplikate:,}",
)
print(
    "Insgesamt entfernte Chunks:",
    f"{insgesamt_entfernt:,}",
)
print(
    "Verbleibende Chunks:",
    f"{len(chunks_df):,}",
)


# ============================================================
# 9. MODELL LADEN
# ============================================================

if LOKALES_MODELL.is_dir():

    print("\nLokal gespeichertes Embedding-Modell wird geladen ...")

    model = SentenceTransformer(
        str(LOKALES_MODELL)
    )

    verwendetes_modell = str(
        LOKALES_MODELL
    )

else:

    print("\nDas Embedding-Modell ist lokal noch nicht vorhanden.")
    print("Es wird jetzt einmalig heruntergeladen ...")
    print("Das kann beim ersten Lauf einige Minuten dauern.")

    model = SentenceTransformer(
        MODEL_NAME
    )

    print("\nModell wird lokal im OpenLens-Projekt gespeichert ...")

    model.save(
        str(LOKALES_MODELL)
    )

    verwendetes_modell = MODEL_NAME

    print("Modell wurde lokal gespeichert.")


embedding_dimension = (
    model.get_sentence_embedding_dimension()
)

print("\nVerwendetes Modell:")
print(verwendetes_modell)

print("\nEmbedding-Dimension:")
print(embedding_dimension)


# ============================================================
# 10. PRÜFEN, OB BEREITS PASSENDE EMBEDDINGS EXISTIEREN
# ============================================================

vorhandene_embeddings_verwenden = False

if (
    not FORCE_REBUILD
    and EMBEDDINGS_NPY.exists()
    and EMBEDDING_METADATA_JSONL.exists()
    and EMBEDDING_MANIFEST.exists()
):

    try:

        with EMBEDDING_MANIFEST.open(
            "r",
            encoding="utf-8",
        ) as datei:

            vorhandenes_manifest = json.load(
                datei
            )


        erwartete_anzahl = int(
            vorhandenes_manifest.get(
                "number_of_embeddings",
                -1,
            )
        )

        erwartete_dimension = int(
            vorhandenes_manifest.get(
                "embedding_dimension",
                -1,
            )
        )

        erwartetes_modell = str(
            vorhandenes_manifest.get(
                "model_name",
                "",
            )
        )


        if (
            erwartete_anzahl == len(chunks_df)
            and erwartete_dimension == embedding_dimension
            and erwartetes_modell == MODEL_NAME
        ):

            vorhandene_embeddings_verwenden = True

            print(
                "\nPassende vorhandene Embeddings wurden gefunden."
            )

        else:

            print(
                "\nVorhandene Embeddings passen nicht mehr "
                "zu den aktuellen Chunks oder zum Modell."
            )

    except Exception as fehler:

        print(
            "\nVorhandene Embedding-Dateien konnten "
            "nicht zuverlässig geprüft werden:"
        )

        print(
            f"{type(fehler).__name__}: {fehler}"
        )


# ============================================================
# 11. EMBEDDINGS ERZEUGEN ODER LADEN
# ============================================================

if vorhandene_embeddings_verwenden:

    print("\nVorhandene Embeddings werden geladen ...")

    embeddings = np.load(
        EMBEDDINGS_NPY
    )

else:

    print("\nEmbeddings werden jetzt erzeugt ...")

    texte = chunks_df[
        "text"
    ].tolist()


    embeddings = model.encode(
        texte,
        batch_size=BATCH_SIZE,
        show_progress_bar=SHOW_PROGRESS_BAR,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )


    embeddings = np.asarray(
        embeddings,
        dtype=np.float32,
    )


    np.save(
        EMBEDDINGS_NPY,
        embeddings,
    )

    print("\nEmbeddings wurden gespeichert.")


# ============================================================
# 12. EMBEDDINGS PRÜFEN
# ============================================================

if embeddings.ndim != 2:

    raise ValueError(
        "\nDie erzeugten Embeddings besitzen nicht "
        "die erwartete zweidimensionale Form.\n"
        f"Aktuelle Form: {embeddings.shape}"
    )


if embeddings.shape[0] != len(chunks_df):

    raise ValueError(
        "\nDie Anzahl der Embeddings stimmt nicht "
        "mit der Anzahl der Chunks überein.\n\n"
        f"Chunks: {len(chunks_df):,}\n"
        f"Embeddings: {embeddings.shape[0]:,}"
    )


if embeddings.shape[1] != embedding_dimension:

    raise ValueError(
        "\nDie Dimension der Embeddings stimmt nicht "
        "mit der Modelldimension überein.\n\n"
        f"Erwartet: {embedding_dimension}\n"
        f"Erhalten: {embeddings.shape[1]}"
    )


if not np.isfinite(
    embeddings
).all():

    raise ValueError(
        "\nDie Embeddings enthalten ungültige Werte "
        "wie NaN oder Unendlich."
    )


print("\nEmbedding-Prüfung erfolgreich.")
print("Embedding-Matrix:", embeddings.shape)
print("Datentyp:", embeddings.dtype)


# ============================================================
# 13. METADATEN FÜR DIE EMBEDDINGS ERSTELLEN
# ============================================================

METADATEN_SPALTEN = [
    spalte
    for spalte in [
        "chunk_id",
        "document_id",
        "chunk_index",
        "title",
        "pdf_name",
        "page_start",
        "page_end",
        "character_count",
        "word_count",
        "site_url",
        "file_url",
        "publicbody",
        "foirequest",
        "uid",
        "source_text_path",
        "clean_text_path",
        "text",
    ]
    if spalte in chunks_df.columns
]


embedding_metadata_df = chunks_df[
    METADATEN_SPALTEN
].copy()


# Die Zeilennummer entspricht exakt der Position
# des Vektors in der NumPy-Matrix.
embedding_metadata_df.insert(
    0,
    "embedding_index",
    np.arange(
        len(embedding_metadata_df),
        dtype=np.int64,
    ),
)


embedding_metadata_df.to_json(
    EMBEDDING_METADATA_JSONL,
    orient="records",
    lines=True,
    force_ascii=False,
)

embedding_metadata_df.to_csv(
    EMBEDDING_METADATA_CSV,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 14. MANIFEST ERSTELLEN
# ============================================================

embedding_dateigroesse_bytes = (
    EMBEDDINGS_NPY.stat().st_size
)

manifest = {
    "project": "OpenLens",
    "model_name": MODEL_NAME,
    "local_model_path": str(
        LOKALES_MODELL.resolve()
    ),
    "number_of_embeddings": int(
        embeddings.shape[0]
    ),
    "embedding_dimension": int(
        embeddings.shape[1]
    ),
    "embedding_dtype": str(
        embeddings.dtype
    ),
    "normalized_embeddings": True,
    "batch_size": BATCH_SIZE,
    "chunk_source": str(
        verwendete_chunk_datei.resolve()
    ),
    "embeddings_file": str(
        EMBEDDINGS_NPY.resolve()
    ),
    "metadata_jsonl": str(
        EMBEDDING_METADATA_JSONL.resolve()
    ),
    "metadata_csv": str(
        EMBEDDING_METADATA_CSV.resolve()
    ),
    "embedding_file_size_bytes": int(
        embedding_dateigroesse_bytes
    ),
    "embedding_file_size_mb": round(
        embedding_dateigroesse_bytes
        / (1024 ** 2),
        2,
    ),
    "created_at": datetime.now(
        timezone.utc
    ).isoformat(),
}


with EMBEDDING_MANIFEST.open(
    "w",
    encoding="utf-8",
) as datei:

    json.dump(
        manifest,
        datei,
        ensure_ascii=False,
        indent=2,
    )


# ============================================================
# 15. ERSTE SEMANTISCHE TESTSUCHE
# ============================================================

print("\n" + "=" * 80)
print("SEMANTISCHE TESTSUCHE")
print("=" * 80)

print("\nSuchanfrage:")
print(TEST_QUERY)


query_embedding = model.encode(
    [TEST_QUERY],
    convert_to_numpy=True,
    normalize_embeddings=True,
)

query_embedding = np.asarray(
    query_embedding,
    dtype=np.float32,
)[0]


# Da sowohl Dokument- als auch Anfragevektoren normalisiert sind,
# entspricht das Skalarprodukt der Kosinus-Ähnlichkeit.
similarity_scores = (
    embeddings
    @ query_embedding
)


top_k_tatsaechlich = min(
    TOP_K,
    len(similarity_scores),
)

top_indices = np.argsort(
    similarity_scores
)[::-1][
    :top_k_tatsaechlich
]


suchergebnis_df = embedding_metadata_df.iloc[
    top_indices
].copy()

suchergebnis_df.insert(
    1,
    "similarity_score",
    similarity_scores[
        top_indices
    ],
)


suchergebnis_df["similarity_score"] = (
    suchergebnis_df[
        "similarity_score"
    ]
    .round(4)
)


suchergebnis_df.to_csv(
    SUCHERGEBNIS_TEST_CSV,
    index=False,
    encoding="utf-8-sig",
)


SUCHERGEBNIS_SPALTEN = [
    spalte
    for spalte in [
        "embedding_index",
        "similarity_score",
        "chunk_id",
        "document_id",
        "title",
        "page_start",
        "page_end",
        "text",
        "site_url",
    ]
    if spalte in suchergebnis_df.columns
]


display(
    suchergebnis_df[
        SUCHERGEBNIS_SPALTEN
    ]
)


# ============================================================
# 16. ABSCHLUSSBERICHT
# ============================================================

print("\n" + "=" * 80)
print("EMBEDDINGS ERFOLGREICH ERSTELLT")
print("=" * 80)

print("\nAnzahl der Chunks:")
print(f"{len(chunks_df):,}")

print("\nAnzahl der Embeddings:")
print(f"{embeddings.shape[0]:,}")

print("\nDimension pro Embedding:")
print(f"{embeddings.shape[1]:,}")

print("\nGröße der Embedding-Datei:")
print(
    f"{embedding_dateigroesse_bytes / (1024 ** 2):,.2f} MB"
)

print("\nLokales Embedding-Modell:")
print(LOKALES_MODELL)

print("\nEmbedding-Matrix:")
print(EMBEDDINGS_NPY)

print("\nEmbedding-Metadaten als JSONL:")
print(EMBEDDING_METADATA_JSONL)

print("\nEmbedding-Metadaten als CSV:")
print(EMBEDDING_METADATA_CSV)

print("\nManifest:")
print(EMBEDDING_MANIFEST)

print("\nTest-Suchergebnis:")
print(SUCHERGEBNIS_TEST_CSV)

print(
    "\nDie Embeddings stehen in der Variable 'embeddings'."
)

print(
    "Die zugehörigen Metadaten stehen in "
    "'embedding_metadata_df'."
)

sentence-transformers ist noch nicht installiert.
Die Installation wird jetzt gestartet ...
sentence-transformers wurde erfolgreich installiert.
OPENLENS: LOKALE EMBEDDINGS ERZEUGEN

Arbeitsordner:
C:\Users\Admin\Desktop\OpenLens

Projektordner:
C:\Users\Admin\Desktop\OpenLens

Chunk-Ordner:
C:\Users\Admin\Desktop\OpenLens\Chunks

Embedding-Ordner:
C:\Users\Admin\Desktop\OpenLens\Embeddings

Lokaler Modellordner:
C:\Users\Admin\Desktop\OpenLens\Modelle\paraphrase-multilingual-MiniLM-L12-v2

Chunk-Daten werden aus JSONL geladen ...

Geladene Chunk-Datei:
C:\Users\Admin\Desktop\OpenLens\Chunks\fragdenstaat_chunks.jsonl

Geladene Chunks:
271

Chunk-Bereinigung:
------------------------------------------------------------
Ursprüngliche Chunks: 271
Entfernte doppelte Chunk-IDs: 0
Insgesamt entfernte Chunks: 0
Verbleibende Chunks: 271

Das Embedding-Modell ist lokal noch nicht vorhanden.
Es wird jetzt einmalig heruntergeladen ...
Das kann beim ersten Lauf einige Minuten dauern.


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

C:\Users\Admin\AppData\Local\Temp\ipykernel_24332\802212418.py:432: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()


Batches:   0%|          | 0/9 [00:00<?, ?it/s]


Embeddings wurden gespeichert.

Embedding-Prüfung erfolgreich.
Embedding-Matrix: (271, 384)
Datentyp: float32

SEMANTISCHE TESTSUCHE

Suchanfrage:
Dokumente über Informationsfreiheit, Behörden und staatliche Entscheidungen


,embedding_index,similarity_score,chunk_id,document_id,title,page_start,page_end,text,site_url
73,73,0.6321,16566300000,165663,Vorbeugegewahrsam,1,2,Landtag Brandenburg\nDrucksache 3/1493\n3. Wah...,https://fragdenstaat.de/dokumente/165663-vorbe...
78,78,0.5419,16592400000,165924,Private Sicherungsdienste,1,2,Landtag Brandenburg\nDrucksache 3/3744\n3. Wah...,https://fragdenstaat.de/dokumente/165924-priva...
40,40,0.5271,16434800000,164348,Elektronische Fußfessel als Haftersatz,1,2,Landtag Brandenburg\nDrucksache 3/138\n3. Wahl...,https://fragdenstaat.de/dokumente/164348-elekt...
161,161,0.5110,16894100000,168941,Transparenzrichtlinie der Europäischen Union,1,2,Landtag Brandenburg\nDrucksache 3/1852\n3. Wah...,https://fragdenstaat.de/dokumente/168941-trans...
270,270,0.5012,21947500000,219475,Behandlung einer Landtagseingabe (3318/3/XI),1,1,Niedersächsicher Landtag – 11. Wahlperiode\nDr...,https://fragdenstaat.de/dokumente/219475-behan...
125,125,0.5007,16751500000,167515,Kita-Kürzungen als Grundgesetzauftrag,1,2,Landtag Brandenburg\nDrucksache 3/922\n3. Wahl...,https://fragdenstaat.de/dokumente/167515-kita-...
261,261,0.5002,17274600000,172746,Begnadigungsrecht,1,1,Landtag Brandenburg\nDrucksache 3/1251\n3. Wah...,https://fragdenstaat.de/dokumente/172746-begna...
50,50,0.4905,16476800000,164768,"""Schwarze Liste"" des Ministeriums der Justiz u...",1,1,Landtag Brandenburg\nDrucksache 3/2352\n3. Wah...,https://fragdenstaat.de/dokumente/164768-schwa...
219,219,0.4905,17124200000,171242,Belastung von Verwaltungsgerichten in Asylsachen,1,2,Landtag Brandenburg\nDrucksache 3/1323\n3. Wah...,https://fragdenstaat.de/dokumente/171242-belas...
169,169,0.4760,16927100000,169271,Erfahrungsbericht der Arbeitsgruppe Asylbetrug...,1,2,Landtag Brandenburg\nDrucksache 3/3263\n3. Wah...,https://fragdenstaat.de/dokumente/169271-erfah...



EMBEDDINGS ERFOLGREICH ERSTELLT

Anzahl der Chunks:
271

Anzahl der Embeddings:
271

Dimension pro Embedding:
384

Größe der Embedding-Datei:
0.40 MB

Lokales Embedding-Modell:
C:\Users\Admin\Desktop\OpenLens\Modelle\paraphrase-multilingual-MiniLM-L12-v2

Embedding-Matrix:
C:\Users\Admin\Desktop\OpenLens\Embeddings\fragdenstaat_embeddings.npy

Embedding-Metadaten als JSONL:
C:\Users\Admin\Desktop\OpenLens\Embeddings\fragdenstaat_embedding_metadata.jsonl

Embedding-Metadaten als CSV:
C:\Users\Admin\Desktop\OpenLens\Embeddings\fragdenstaat_embedding_metadata.csv

Manifest:
C:\Users\Admin\Desktop\OpenLens\Embeddings\fragdenstaat_embedding_manifest.json

Test-Suchergebnis:
C:\Users\Admin\Desktop\OpenLens\Embeddings\semantic_search_test.csv

Die Embeddings stehen in der Variable 'embeddings'.
Die zugehörigen Metadaten stehen in 'embedding_metadata_df'.
